In [0]:
# Agent3: Visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load silver dataset
df = spark.sql("SELECT * FROM samplesuperstore.silverdata.orders").toPandas()
print("Dataset loaded with shape:", df.shape)
df.head()


In [0]:
plt.figure(figsize=(8,6))
sns.barplot(x="Category", y="Sales", data=df, estimator=sum, color="skyblue")
sns.barplot(x="Category", y="Profit", data=df, estimator=sum, color="orange")
plt.title("Sales vs Profit by Category")
plt.show()


In [0]:
plt.figure(figsize=(8,6))
sns.barplot(x="Region", y="Sales", data=df, estimator=sum, color="green")
plt.title("Sales by Region")
plt.show()

plt.figure(figsize=(8,6))
sns.barplot(x="Region", y="Profit", data=df, estimator=sum, color="red")
plt.title("Profit by Region")
plt.show()


In [0]:
# Pivot sales & profit by Stateprovince
pivot = df.pivot_table(values=["Sales","Profit"], index="Stateprovince", aggfunc="sum")

plt.figure(figsize=(12,8))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues")
plt.title("Sales & Profit by Stateprovince")
plt.show()


In [0]:
plt.figure(figsize=(8,6))
sns.scatterplot(x="Discount", y="Profit", data=df, hue="Category")
plt.title("Discount vs Profit")
plt.show()


In [0]:
# Top 5 states by sales
top_states_sales = (
    df.groupby("Stateprovince")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(5)
)

print("Top 5 States by Sales:")
print(top_states_sales)


In [0]:
%sql
USE CATALOG samplesuperstore;

CREATE OR REPLACE FUNCTION silverdata.run_visualization(dataset STRING)
RETURNS STRING
LANGUAGE PYTHON
AS $$
def run_visualization(dataset: str) -> str:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    df = spark.table(dataset)
    top_categories = df.groupBy("Category").count().orderBy("count", ascending=False).limit(3)

    result = [f"{row['Category']}: {row['count']}" for row in top_categories.collect()]
    return "Top Categories → " + ", ".join(result)
$$;


In [0]:
%sql
-- Step 1: Switch to the correct catalog
USE CATALOG samplesuperstore;

-- Step 2: Call the UC function with silverdata.orders table
SELECT silverdata.run_data_cleaning('silverdata.orders');

-- Step 3: Verify function registration (optional check)
SHOW FUNCTIONS IN silverdata;
